# 🔥 Notebook 7: Feed-Forward Networks

**The "Thinking" Component of Transformers**

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. Understand **why feed-forward networks** are needed
2. Implement **position-wise transformations**
3. Learn about **expansion and projection**
4. Understand **non-linearity** (ReLU, GELU)
5. Build a complete **FeedForward layer**
6. Analyze **computational complexity**

---

## 📚 Table of Contents

1. [The Attention Limitation](#1-the-attention-limitation)
2. [Feed-Forward Intuition](#2-feed-forward-intuition)
3. [Position-Wise Transformation](#3-position-wise-transformation)
4. [Expansion and Projection](#4-expansion-and-projection)
5. [Activation Functions](#5-activation-functions)
6. [Complete Feed-Forward Network](#6-complete-feed-forward-network)
7. [Computational Analysis](#7-computational-analysis)
8. [Key Takeaways](#8-key-takeaways)

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
import os

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

### Environment Setup (Colab/Local)

This cell detects whether you're running in Google Colab or locally and sets up the environment accordingly.

In [ ]:
# ========================================
# ENVIRONMENT DETECTION & SETUP
# ========================================

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

# Setup directories
if IN_COLAB:
    # Create necessary directories for Colab
    os.makedirs('data', exist_ok=True)
    os.makedirs('visualizations/tokenization', exist_ok=True)
    os.makedirs('visualizations/embeddings', exist_ok=True)
    os.makedirs('visualizations/data_pipeline', exist_ok=True)
    print("Created directories")
    
    # Set paths for Colab
    DATA_DIR = 'data'
    VIZ_DIR = 'visualizations'
else:
    # Use relative paths for local execution
    DATA_DIR = '../data'
    VIZ_DIR = '../visualizations'
    
    # Create directories if they don't exist (LOCAL FIX)
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/tokenization', exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/embeddings', exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/data_pipeline', exist_ok=True)
    print(f"Created directories: {DATA_DIR}, {VIZ_DIR}")

print(f"\nData directory: {DATA_DIR}")
print(f"Visualization directory: {VIZ_DIR}")

In [ ]:
# Load configuration
with open(f'{DATA_DIR}/data_config.pkl', 'rb') as f:
    config = pickle.load(f)

vocab_size = config['vocab_size']
block_size = config['block_size']
batch_size = config['batch_size']

# Load embedding config
checkpoint = torch.load(f'{DATA_DIR}/embedding_layer.pth')
n_embd = checkpoint['n_embd']

print(f"📊 Configuration:")
print(f"Vocabulary size: {vocab_size}")
print(f"Block size:      {block_size}")
print(f"Embedding dim:   {n_embd}")

---

## 1. The Attention Limitation

### 🚨 The Problem

**Attention is great at gathering information, but...**

```
Attention:
- Aggregates information from other tokens
- Weighted sum of values
- Linear operation (no non-linearity!)
```

**What's missing?**
- **No individual processing**: Each token can't "think" about what it gathered
- **No non-linearity**: Can't learn complex patterns
- **Limited capacity**: Just weighted averaging

### 💡 The Solution: Feed-Forward Networks

After attention gathers information, **Feed-Forward Networks** allow each token to:
1. **Process** the gathered information
2. **Transform** it non-linearly
3. **Extract** higher-level features

**Key Insight**: Attention = "communication", FFN = "thinking"

---

## 2. Feed-Forward Intuition

### 🧠 The Two-Stage Process

Think of a **research paper review**:

**Stage 1: Attention (Gathering)**
- Read related papers
- Collect relevant information
- Aggregate different perspectives

**Stage 2: Feed-Forward (Thinking)**
- Analyze the gathered information
- Form your own opinion
- Synthesize new insights

### 📐 Mathematical Formulation

```
FFN(x) = W_2 @ ReLU(W_1 @ x + b_1) + b_2
```

Or more commonly:
```
FFN(x) = Linear_2(GELU(Linear_1(x)))
```

**Two linear layers with non-linearity in between!**

---

## 3. Position-Wise Transformation

### 🎯 Key Property

FFN is applied **independently** to each position:

```
Input:  (B, T, C)
        ↓
Apply FFN to each of T positions separately
        ↓
Output: (B, T, C)
```

**Same FFN weights for all positions!**

### 💻 Simple Example

In [ ]:
# Create sample input
B, T, C = 4, 8, n_embd
x = torch.randn(B, T, C)

print(f"📊 Input shape: {x.shape}")
print(f"   Batch (B):    {B}")
print(f"   Sequence (T): {T}")
print(f"   Channels (C): {C}")

# Simple linear layer (position-wise)
linear = nn.Linear(C, C)
output = linear(x)

print(f"\n📤 Output shape: {output.shape}")
print(f"\n💡 Same transformation applied to all {T} positions!")

---

## 4. Expansion and Projection

### 🎯 The Standard Architecture

Transformers use a **two-layer FFN** with expansion:

```
Input:  (B, T, d_model)
   ↓
Expand: (B, T, 4 × d_model)  ← Expansion factor (usually 4)
   ↓
Non-linearity (GELU/ReLU)
   ↓
Project: (B, T, d_model)     ← Back to original size
```

**Why expand?**
- More capacity for learning
- Richer feature space
- Better performance

### 💻 Implementation

In [ ]:
# Hyperparameters
expansion_factor = 4
d_ff = n_embd * expansion_factor  # Feed-forward dimension

print(f"📊 Feed-Forward Configuration:")
print(f"Input dimension:  {n_embd}")
print(f"Expansion factor: {expansion_factor}")
print(f"Hidden dimension: {d_ff}")
print(f"Output dimension: {n_embd}")

In [ ]:
# Create two linear layers
fc1 = nn.Linear(n_embd, d_ff)  # Expansion
fc2 = nn.Linear(d_ff, n_embd)  # Projection

# Forward pass
x = torch.randn(B, T, n_embd)

# Expand
hidden = fc1(x)
print(f"📥 Input:  {x.shape}")
print(f"🔄 Hidden: {hidden.shape}  ← Expanded by {expansion_factor}×")

# Project back
output = fc2(hidden)
print(f"📤 Output: {output.shape}  ← Projected back")

print(f"\n💡 Shape preserved: (B, T, C) → (B, T, C)")

### 📊 Visualize Expansion

In [ ]:
# Visualize the expansion and projection
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Input
im1 = axes[0].imshow(x[0].detach().numpy(), cmap='viridis', aspect='auto')
axes[0].set_title(f'Input\n(T={T}, C={n_embd})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Embedding Dim', fontsize=11)
axes[0].set_ylabel('Sequence Position', fontsize=11)
plt.colorbar(im1, ax=axes[0])

# Hidden (expanded)
im2 = axes[1].imshow(hidden[0].detach().numpy(), cmap='plasma', aspect='auto')
axes[1].set_title(f'Hidden (Expanded)\n(T={T}, C={d_ff})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hidden Dim', fontsize=11)
axes[1].set_ylabel('Sequence Position', fontsize=11)
plt.colorbar(im2, ax=axes[1])

# Output
im3 = axes[2].imshow(output[0].detach().numpy(), cmap='coolwarm', aspect='auto')
axes[2].set_title(f'Output (Projected)\n(T={T}, C={n_embd})', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Embedding Dim', fontsize=11)
axes[2].set_ylabel('Sequence Position', fontsize=11)
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/embeddings/ffn_expansion.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 Middle layer is 4× wider for more capacity!")

---

## 5. Activation Functions

### 🎯 Why Non-Linearity?

Without activation functions:
```
Linear_2(Linear_1(x)) = (W_2 @ W_1) @ x = W_combined @ x
```

**Just another linear transformation!** No benefit from depth.

With activation:
```
Linear_2(σ(Linear_1(x)))  ← Non-linear!
```

### 📊 Common Activation Functions

1. **ReLU**: `max(0, x)`
2. **GELU**: Gaussian Error Linear Unit (smoother, used in GPT)
3. **SiLU/Swish**: `x * sigmoid(x)`

In [ ]:
# Compare activation functions
x_range = torch.linspace(-3, 3, 1000)

# ReLU
relu = F.relu(x_range)

# GELU
gelu = F.gelu(x_range)

# SiLU (Swish)
silu = F.silu(x_range)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ReLU
axes[0].plot(x_range.numpy(), relu.numpy(), linewidth=2, color='steelblue')
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[0].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[0].set_title('ReLU\nmax(0, x)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('x', fontsize=11)
axes[0].set_ylabel('ReLU(x)', fontsize=11)
axes[0].grid(alpha=0.3)

# GELU
axes[1].plot(x_range.numpy(), gelu.numpy(), linewidth=2, color='coral')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[1].set_title('GELU (GPT)\nSmooth approximation', fontsize=13, fontweight='bold')
axes[1].set_xlabel('x', fontsize=11)
axes[1].set_ylabel('GELU(x)', fontsize=11)
axes[1].grid(alpha=0.3)

# SiLU
axes[2].plot(x_range.numpy(), silu.numpy(), linewidth=2, color='green')
axes[2].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[2].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[2].set_title('SiLU/Swish\nx * sigmoid(x)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('x', fontsize=11)
axes[2].set_ylabel('SiLU(x)', fontsize=11)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/embeddings/activation_functions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 GELU is smoother than ReLU, better for gradients!")

### 🔍 Apply Activation

In [ ]:
# Test with activation
x = torch.randn(B, T, n_embd)

# Forward pass with GELU
hidden = fc1(x)
activated = F.gelu(hidden)
output = fc2(activated)

print(f"📊 Forward Pass with GELU:")
print(f"Input:     {x.shape}")
print(f"Hidden:    {hidden.shape}")
print(f"Activated: {activated.shape}")
print(f"Output:    {output.shape}")

print(f"\n📈 Statistics:")
print(f"Hidden (before GELU): mean={hidden.mean():.4f}, std={hidden.std():.4f}")
print(f"Activated (after GELU): mean={activated.mean():.4f}, std={activated.std():.4f}")

---

## 6. Complete Feed-Forward Network

Let's implement the complete FFN layer:

In [ ]:
class FeedForward(nn.Module):
    """
    Position-wise feed-forward network.
    
    Args:
        n_embd: Embedding dimension
        expansion_factor: Hidden layer expansion (default: 4)
        dropout: Dropout probability
    """
    
    def __init__(self, n_embd, expansion_factor=4, dropout=0.1):
        super().__init__()
        
        d_ff = n_embd * expansion_factor
        
        self.net = nn.Sequential(
            nn.Linear(n_embd, d_ff),      # Expansion
            nn.GELU(),                     # Non-linearity
            nn.Linear(d_ff, n_embd),       # Projection
            nn.Dropout(dropout)            # Regularization
        )
    
    def forward(self, x):
        """
        Args:
            x: Input, shape (B, T, C)
        
        Returns:
            out: Output, shape (B, T, C)
        """
        return self.net(x)

# Create feed-forward network
ffn = FeedForward(n_embd, expansion_factor=4)

print(f"🔥 Feed-Forward Network Created!")
print(f"\n📊 Architecture:")
print(ffn)

# Count parameters
total_params = sum(p.numel() for p in ffn.parameters())
print(f"\n💾 Total parameters: {total_params:,}")

In [ ]:
# Test the feed-forward network
test_input = torch.randn(4, 8, n_embd)
test_output = ffn(test_input)

print(f"📥 Input shape:  {test_input.shape}")
print(f"📤 Output shape: {test_output.shape}")
print(f"\n✅ Feed-forward network preserves shape: (B, T, C) → (B, T, C)")

### 📊 Visualize Transformation

In [ ]:
# Visualize input vs output
with torch.no_grad():
    x_vis = torch.randn(1, 8, n_embd)
    y_vis = ffn(x_vis)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Input
im1 = axes[0].imshow(x_vis[0].numpy(), cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
axes[0].set_title('Input', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Embedding Dimension', fontsize=11)
axes[0].set_ylabel('Sequence Position', fontsize=11)
plt.colorbar(im1, ax=axes[0])

# Output
im2 = axes[1].imshow(y_vis[0].numpy(), cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
axes[1].set_title('Output (After FFN)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Embedding Dimension', fontsize=11)
axes[1].set_ylabel('Sequence Position', fontsize=11)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/embeddings/ffn_transformation.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 FFN transforms each position independently!")

---

## 7. Computational Analysis

### 📊 Parameter Count

In [ ]:
# Analyze parameters
d = n_embd
d_ff = d * expansion_factor

print("📊 Parameter Breakdown:")
print("=" * 60)

# First linear layer
fc1_params = d * d_ff + d_ff  # Weights + biases
print(f"\nFirst Linear (Expansion):")
print(f"   Weights: {d} × {d_ff} = {d * d_ff:,}")
print(f"   Biases:  {d_ff:,}")
print(f"   Total:   {fc1_params:,}")

# Second linear layer
fc2_params = d_ff * d + d  # Weights + biases
print(f"\nSecond Linear (Projection):")
print(f"   Weights: {d_ff} × {d} = {d_ff * d:,}")
print(f"   Biases:  {d:,}")
print(f"   Total:   {fc2_params:,}")

# Total
print(f"\n{'='*60}")
print(f"Total Parameters: {total_params:,}")
print(f"{'='*60}")

print(f"\n💡 Most parameters are in the weight matrices!")

### ⚡ Computational Complexity

In [ ]:
# Analyze computational complexity
T = block_size
d = n_embd
d_ff = d * expansion_factor

print("⚡ Computational Complexity:")
print("=" * 60)

print(f"\nGiven:")
print(f"   Sequence length (T): {T}")
print(f"   Embedding dim (d):   {d}")
print(f"   Hidden dim (d_ff):   {d_ff}")

print(f"\nFirst Linear Layer:")
print(f"   O(T × d × d_ff) = O({T} × {d} × {d_ff})")

print(f"\nActivation (GELU):")
print(f"   O(T × d_ff) = O({T} × {d_ff})")

print(f"\nSecond Linear Layer:")
print(f"   O(T × d_ff × d) = O({T} × {d_ff} × {d})")

print(f"\nTotal Complexity:")
print(f"   O(T × d × d_ff) = O(T × d²) since d_ff = 4d")

print(f"\n💡 Linear in sequence length T!")
print(f"   (Unlike attention which is O(T²))")

### 📈 Compare with Attention

In [ ]:
# Compare FFN vs Attention complexity
sequence_lengths = [64, 128, 256, 512, 1024, 2048]

# Attention: O(T² × d)
attention_ops = [T**2 * d for T in sequence_lengths]

# FFN: O(T × d²)
ffn_ops = [T * d**2 for T in sequence_lengths]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(sequence_lengths, attention_ops, 'o-', linewidth=2, markersize=8, 
        label='Attention: O(T² × d)', color='steelblue')
ax.plot(sequence_lengths, ffn_ops, 's-', linewidth=2, markersize=8, 
        label='FFN: O(T × d²)', color='coral')

ax.set_xlabel('Sequence Length (T)', fontsize=12, fontweight='bold')
ax.set_ylabel('Operations (arbitrary units)', fontsize=12, fontweight='bold')
ax.set_title('Computational Complexity: Attention vs FFN', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/embeddings/complexity_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n💡 For long sequences, Attention dominates!")
print(f"   At T=2048: Attention is {attention_ops[-1]/ffn_ops[-1]:.1f}× more expensive")

---

## 8. Key Takeaways

### ✅ What We Learned

1. **Why Feed-Forward Networks?**
   - Attention gathers information (communication)
   - FFN processes information (thinking)
   - Adds non-linearity and capacity
   - Essential for learning complex patterns

2. **Architecture**
   ```
   Input (B, T, d)
      ↓
   Linear: d → 4d (Expansion)
      ↓
   GELU (Non-linearity)
      ↓
   Linear: 4d → d (Projection)
      ↓
   Dropout
      ↓
   Output (B, T, d)
   ```

3. **Position-Wise**
   - Same FFN applied to each position
   - No interaction between positions
   - Parallel processing

4. **Expansion Factor**
   - Typically 4× (GPT, BERT)
   - Larger = more capacity
   - Trade-off: capacity vs computation

5. **Activation Functions**
   - GELU: Smooth, used in GPT
   - ReLU: Simple, fast
   - Critical for non-linearity

6. **Computational Complexity**
   - O(T × d²) - Linear in sequence length
   - Cheaper than attention for long sequences
   - But still significant (most parameters here!)

### 🎯 Key Insights

- **Complementary to Attention**: Attention = gather, FFN = process
- **Most Parameters**: ~2/3 of Transformer parameters are in FFN
- **Position-Wise**: Each token processed independently
- **Non-Linearity**: Essential for learning complex functions

### 🔮 What's Next?

In **Notebook 8**, we'll combine everything into a **Complete Transformer Block**:
- Multi-head attention + Feed-forward
- Residual connections
- Layer normalization
- The full building block of GPT!

In [ ]:
# Save feed-forward network
torch.save({
    'feed_forward': ffn.state_dict(),
    'n_embd': n_embd,
    'expansion_factor': expansion_factor
}, f'{DATA_DIR}/feed_forward.pth')

print("✅ Feed-forward network saved to f'{DATA_DIR}/feed_forward.pth'")